# Testing the draft 2D histogram (`team/new_graphics/hist2d.py`)

This notebook exercises `make_hist2d()`, the draft implementation of the new 2D
histogram plot type from the approved Task 1E prototype. It is **not** wired into
`.plot()` yet — this is for visually checking the draft before integration.

**How to run:** open this notebook from the `team/new_graphics/new_graphics_demos/`
folder using the `symbulate` conda environment, then Run All.

**What to look for in every plot:**
- Filled bin mesh in **viridis** (the package's sequential colormap, from `symbulate.mplstyle` — no hardcoded `Blues`)
- Colorbar on the right, labeled "Density" (or "Count" when `normalize=False`), placed with `make_axes_locatable`
- Color scale starts at 0, so empty bins read as "no data"
- Axis labels read "X" and "Y"; title reads "2-D Histogram"
- No reference grid (the mesh covers the axes)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# This notebook lives in <repo>/team/new_graphics/new_graphics_demos/.
REPO_ROOT = Path.cwd().parents[2]

# Make `import symbulate` find the dev copy in this repo (not an older
# installed copy in site-packages), and `from hist2d import ...` find
# the draft module one folder up.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(Path.cwd().parent))

# The draft helper and its per-plot-type constants
from hist2d import make_hist2d, HIST2D_DEFAULT_BINS, HIST2D_OVERLAY_WARNING

# Apply the global style file (viridis colormap, fonts, spines).
# Not yet wired into the package, so we apply it by hand here.
STYLE_PATH = REPO_ROOT / "symbulate" / "symbulate.mplstyle"
plt.style.use(STYLE_PATH)

rng = np.random.default_rng(7)
print(f"HIST2D_DEFAULT_BINS = {HIST2D_DEFAULT_BINS}")

## 1. Reproduce the approved prototype

Correlated bivariate normal data — this should match the prototype image:
a tilted elliptical blob, bright yellow at the dense center fading through
green and blue to dark purple where there is no data.

In [ ]:
xy = rng.multivariate_normal([0, 0], [[2.0, 1.2], [1.2, 2.0]], size=3000)

ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax)
plt.show()  # title, axis labels, and colorbar are set by make_hist2d

## 2. Options: counts, custom bins

`normalize=False` should switch the colorbar label to "Count".

In [ ]:
ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax, normalize=False)
plt.show()

In [ ]:
# Coarser bins -- each cell aggregates more simulations
ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax, bins=12)
plt.show()

## 3. Hexagon bins — `hex=True`

The same plot with hexagonal bins instead of squares; the title switches to
"Hexbin Plot". Everything else (viridis, colorbar, `normalize`, overlay
warning) behaves identically. matplotlib's `hexbin` has no `density` option,
so `make_hist2d` rescales the counts by hand — hexagon volumes
(density × cell area) sum to 1, just like the square-bin version.

In [ ]:
ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax, hex=True)
plt.show()

In [ ]:
# Hexbin with raw counts -- colorbar switches to "Count"
ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax, hex=True, normalize=False)
plt.show()

## 4. Overlay behavior — the readability *warning* category

Per the overlay policy, two 2D histograms on the same axes is one of the
designated **warning** cases: the second histogram still draws (overlay is
never blocked outside the `'marginal'` hard-error case), but its color scale
competes with the first, so a student-friendly warning prints below the plot.

The exact warning wording is still an open decision in `DECISIONS.md` — the
text printed here is the draft wording from `hist2d.py`.

In [ ]:
ax = plt.gca()
make_hist2d(xy[:, 0], xy[:, 1], ax)
make_hist2d(xy[:, 0] + 3, xy[:, 1] + 3, ax)  # draws, and prints the warning
plt.show()

## 5. With real symbulate simulation data

Same prototype, but the data comes from an actual `RV(BivariateNormal(...)).sim(...)`
call instead of raw numpy.

Caveat while testing today: creating `RVResults` currently calls `init_color()`,
which resets the color cycle to tab10, and importing `symbulate` switches the
matplotlib style back to the old stylesheet (both Phase 2 items) — so we re-apply
the style *after* importing and simulating.

In [ ]:
from symbulate import RV, BivariateNormal

X = RV(BivariateNormal(mean1=0, mean2=0, sd1=1.4, sd2=1.4, corr=0.6))
sims = X.sim(3000)

plt.style.use(STYLE_PATH)  # re-apply: import + RVResults reset the style

arr = np.asarray(sims.results)
ax = plt.gca()
make_hist2d(arr[:, 0], arr[:, 1], ax)
plt.show()

## 6. Variables on very different ranges

The two axes need not share a scale. Here `X` is roughly in [-3, 3] while `Y` is
centered near 20 with a much wider spread. `hist2d` bins each axis over its **own**
range independently — `bins` is a count per axis, not a shared grid spacing — so
the mesh fills the axes box regardless of the mismatch, with no manual rescaling.
The tick labels on each axis reflect its own range.

In [ ]:
x_narrow = rng.normal(0, 1, 4000)    # roughly [-3, 3]
y_wide = rng.normal(20, 8, 4000)     # centered near 20, much wider spread

ax = plt.gca()
make_hist2d(x_narrow, y_wide, ax)
plt.show()  # each axis bins over its own range; the mesh fills the box

## Verification checklist

- [ ] Section 1 matches the approved prototype image
- [ ] Viridis colormap (not Blues); empty bins are dark purple, dense bins yellow
- [ ] Colorbar on the right, sized proportionally to the axes
- [ ] Colorbar reads "Density"; `normalize=False` switches it to "Count"
- [ ] Color scale starts at 0
- [ ] Axis labels "X" and "Y"; title "2-D Histogram"
- [ ] No grid fragments around the mesh
- [ ] `bins=` override works
- [ ] `hex=True` bins into hexagons and switches the title to "Hexbin Plot"
- [ ] Hexbin honors `normalize` (Density colorbar) and `normalize=False` (Count)
- [ ] Single 2D histogram prints no warning
- [ ] Second 2D histogram on the same axes draws AND prints the readability warning
- [ ] Works on real `.sim()` output, not just numpy arrays
- [ ] Section 6: axes on very different ranges (X ~ [-3, 3], Y ~ near 20) each bin over their own range; mesh fills the box